# RAG chunk optimize — results strengthening (①②)

Two post-completion analyses that harden the portfolio's conclusions. Self-contained — **Runtime → Run all** after selecting a GPU runtime.

1. **① Effect-size regression + minimum detectable effect** (`scripts/20_effect_size.py`, CPU) — "size dominates, method ties" as a coefficient table + proof the tie is not underpowered.
2. **② Reranker cross-dataset transfer** (`scripts/21_reranker_transfer.py`, **GPU**) — does the Stage 8 NQ-tuned reranker's +0.107 R@1 gain survive on TriviaQA?

> **One-time retrain (~17 min)** happens automatically before ② if the Stage 8 weights are missing from Drive (they were lost). Training data is already on Drive.

**Before running:** Runtime → Change runtime type → **GPU (T4)**, then Runtime → **Run all**. Authorize Google Drive when the setup cell prompts. The dependency cell installs faiss etc. (~1 min).

## Setup — mount Drive, set paths (edit `PROJECT_DIR` only if your folder differs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
# Folder you uploaded (the project root that contains config.py). Edit if needed.
PROJECT_DIR = '<project-root>'
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print(C.summary())

## Install dependencies (faiss, datasets, sentence-transformers, …)

Colab ships torch but not faiss; this installs the project's `requirements.txt`. ~1 min.

In [ ]:
!pip install -q -r requirements.txt

## ① Effect-size & MDE (CPU, fast — reads archives only, reruns nothing)

Writes `artifacts/results/portfolio/effect_size_{report.md, coefficients.csv, tornado.png}`.

In [ ]:
!python scripts/20_effect_size.py

## Ensure the fine-tuned reranker exists (auto-retrain if Drive weights were lost)

Runs only if `bge_reranker_ft/final` has no weights file. ~17 min on T4; idempotent.

In [ ]:
# Ensure the fine-tuned reranker WEIGHTS exist before ②.
# The Stage 8 run's ~1.1 GB safetensors never durably persisted to Drive
# (only config.json + tokenizer did), so ② would otherwise load a headless
# model. Retrain (~17 min on T4) ONLY if the weights file is actually missing;
# the training data (stage8_train_groups.jsonl) is already on Drive, so this
# does NOT need script 16. Idempotent: if weights are present it just skips.
import os, sys, pathlib, subprocess
ft = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'models' / 'bge_reranker_ft' / 'final'
def _weights(d):
    return list(d.glob('*.safetensors')) + list(d.glob('pytorch_model.bin'))
w = _weights(ft)
if w:
    print('FT reranker weights present:', [f.name for f in w])
else:
    print('FT reranker weights MISSING at', ft)
    print('Retraining scripts/17_train_reranker.py (~17 min on T4)...')
    r = subprocess.run([sys.executable, 'scripts/17_train_reranker.py'])
    if r.returncode != 0:
        raise SystemExit('reranker training failed — see output above')
    w = _weights(ft)
    assert w, f'training finished but no weights file in {ft}'
    print('Retrained OK. weights now:', [f.name for f in w])

## ② Reranker cross-dataset transfer (GPU)

Same 5 configs / 3 arms / shared BGE top-20 pool as Stage 8; only the eval dataset changes (TriviaQA). Resume-safe (per-config checkpoint). Writes `artifacts/results/latest/stage8_transfer_*`.

In [ ]:
!python scripts/21_reranker_transfer.py

## Review ② result inline

In [ ]:
# Review the transfer result inline
from IPython.display import Image, Markdown, display
import pathlib, os
latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
display(Markdown((latest / 'stage8_transfer_summary.md').read_text(encoding='utf-8')))
display(Image(str(latest / 'stage8_transfer_delta.png')))

## Outputs & next step

- ① → `artifacts/results/portfolio/effect_size_report.md` (+ CSV, tornado PNG)
- ② → `artifacts/results/latest/stage8_transfer_summary.md` (+ CSV, delta PNG)

Read the printed **VERDICT** from ② (TRANSFERS / DIRECTIONAL / NO-TRANSFER). Record the ② console output, then update README / docs and the project journal accordingly.

> If ② retrained the reranker: cross-dataset numbers use the retrained model (seed 42, same recipe); the in-domain column is the original Stage 8 archive — they match within training noise. Let Drive finish syncing the new ~1.1 GB weights so they persist.